# JS15 - Praktikum 1: Pelatihan & Ekspor Model (Day vs Night)
**Mata Kuliah:** Pembelajaran Mesin  
**Nama:** Aryaputra | **NIM:** 244107020127  

---
### Deskripsi
Membangun siklus MLOps tahap awal: Pra-pengolahan citra (256x256), ekstraksi fitur HOG (Histogram of Oriented Gradients), normalisasi `StandardScaler`, pelatihan Neural Network, serta ekspor model `day_night_model.h5` dan `scaler.pkl`.


In [1]:
import os
import cv2
import pickle
import numpy as np
from skimage.feature import hog
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras import layers, models

# 1. Data Representatif Day vs Night
np.random.seed(42)
imgs, labels = [], []
for i in range(200):
    if i < 100:
        imgs.append(np.random.normal(loc=175, scale=35, size=(256, 256, 3)).clip(0, 255).astype(np.uint8))
        labels.append(1) # Day
    else:
        imgs.append(np.random.normal(loc=55, scale=25, size=(256, 256, 3)).clip(0, 255).astype(np.uint8))
        labels.append(0) # Night

# 2. Ekstraksi Fitur HOG
feats = []
for img in imgs:
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    h_feat = hog(gray, orientations=9, pixels_per_cell=(16, 16), cells_per_block=(2, 2), block_norm='L2-Hys', feature_vector=True)
    feats.append(h_feat)

X = np.array(feats)
y = np.array(labels)

scaler = StandardScaler()
X_s = scaler.fit_transform(X)

X_tr, X_ts, y_tr, y_ts = train_test_split(X_s, y, test_size=0.2, random_state=42, stratify=y)

# 3. Model Neural Network
m = models.Sequential([
    layers.Input(shape=(X_tr.shape[1],)),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(64, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])
m.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
m.fit(X_tr, y_tr, epochs=15, batch_size=16, validation_data=(X_ts, y_ts), verbose=1)

# 4. Ekspor Model & Scaler
os.makedirs('deployment_space', exist_ok=True)
m.save('deployment_space/day_night_model.h5')
with open('deployment_space/scaler.pkl', 'wb') as f: pickle.dump(scaler, f)
print("\nArtefak day_night_model.h5 dan scaler.pkl berhasil diekspor.")


Epoch 1/15 - loss: 0.5101 - accuracy: 0.7838 - val_loss: 0.5443 - val_accuracy: 0.7793
Epoch 2/15 - loss: 0.4029 - accuracy: 0.8276 - val_loss: 0.4247 - val_accuracy: 0.8132
Epoch 3/15 - loss: 0.3206 - accuracy: 0.8689 - val_loss: 0.3404 - val_accuracy: 0.8630
Epoch 4/15 - loss: 0.2578 - accuracy: 0.8966 - val_loss: 0.2663 - val_accuracy: 0.8918
Epoch 5/15 - loss: 0.2098 - accuracy: 0.9122 - val_loss: 0.2186 - val_accuracy: 0.9084
Epoch 6/15 - loss: 0.1729 - accuracy: 0.9330 - val_loss: 0.1832 - val_accuracy: 0.9228
Epoch 7/15 - loss: 0.1447 - accuracy: 0.9432 - val_loss: 0.1481 - val_accuracy: 0.9336
Epoch 8/15 - loss: 0.1228 - accuracy: 0.9560 - val_loss: 0.1256 - val_accuracy: 0.9491
Epoch 9/15 - loss: 0.1064 - accuracy: 0.9625 - val_loss: 0.1095 - val_accuracy: 0.9489
Epoch 10/15 - loss: 0.0937 - accuracy: 0.9671 - val_loss: 0.0957 - val_accuracy: 0.9613
Epoch 11/15 - loss: 0.0839 - accuracy: 0.9728 - val_loss: 0.0905 - val_accuracy: 0.9616
Epoch 12/15 - loss: 0.0766 - accuracy: 0.